**Setup and Baseline Test**

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(os.path.abspath('..'))

from src.preprocessing import load_and_preprocess_data
from src.models import get_baseline_models
from src.evaluation import evaluate_model
from src.drift_detection import detect_feature_drift
from src.active_learning import get_hitl_queue_metrics

SyntaxError: invalid syntax (84874469.py, line 13)

In [ ]:
X_train, y_train, X_val, y_val, X_test, y_test = load_and_preprocess_data('../data/creditcard.csv')

# Train our best model (Random Forest)
models = get_baseline_models()
model = models['RandomForest']
model.fit(X_train, y_train)

# Evaluate on the untouched Test set (Normal Future)
y_pred_proba = model.predict_proba(X_test)[:, 1]
metrics_normal = evaluate_model(y_test, y_pred_proba)

print(f"Normal Future - False Negatives (Missed Fraud): {metrics_normal['false_negatives']}")
print(f"Normal Future - True Positives (Caught Fraud): {metrics_normal['true_positives']}")

Normal Future - False Negatives (Missed Fraud): 19
Normal Future - True Positives (Caught Fraud): 33


**Simulating the Adversarial Attack (Concept Drift)**

In [ ]:
X_test_drifted = X_test.copy()

# Identify where the actual fraud is in the test set
fraud_indices = y_test[y_test == 1].index

# SIMULATION: Scammers figure out how to bypass our V4, V11, and V14 thresholds
# We add a massive statistical shift to these features, but ONLY for the fraudulent transactions
X_test_drifted.loc[fraud_indices, 'V4'] -= 5.0
X_test_drifted.loc[fraud_indices, 'V11'] -= 3.0
X_test_drifted.loc[fraud_indices, 'V14'] += 5.0

print("Adversarial attack simulated. Scammers have shifted their behavior on V4, V11, and V14.")

Adversarial attack simulated. Scammers have shifted their behavior on V4, V11, and V14.


**The Model Fails**

In [ ]:
# Evaluate on the Drifted Test set
y_pred_proba_drifted = model.predict_proba(X_test_drifted)[:, 1]
metrics_drifted = evaluate_model(y_test, y_pred_proba_drifted)

print("\n--- MODEL PERFORMANCE POST-DRIFT ---")
print(f"Missed Fraud Before Attack: {metrics_normal['false_negatives']}")
print(f"Missed Fraud AFTER Attack:  {metrics_drifted['false_negatives']}")
print(f"Caught Fraud Before Attack: {metrics_normal['true_positives']}")
print(f"Caught Fraud AFTER Attack:  {metrics_drifted['true_positives']}")

print("\nConclusion:")
print("The model is completely blind to the new attack vector. Financial losses are skyrocketing.")


--- MODEL PERFORMANCE POST-DRIFT ---
Missed Fraud Before Attack: 19
Missed Fraud AFTER Attack:  38
Caught Fraud Before Attack: 33
Caught Fraud AFTER Attack:  14

Conclusion:
The model is completely blind to the new attack vector. Financial losses are skyrocketing.


**Automated Drift Detection**

In [ ]:
# We use our Validation set as the "Reference" (what the model knows)
# and compare it to the "Current" drifted test set
print("Running Statistical Drift Detection (KS-Test)...")
drift_report, features_drifted = detect_feature_drift(X_val, X_test_drifted, p_value_threshold=0.01)

print(f"\nSYSTEM ALERT: {features_drifted} features have significantly drifted from the baseline distribution.")

if features_drifted > 0:
    print("\nDrifted Features Details:")
    for feature, metrics in drift_report.items():
        if metrics['is_drifting']:
            print(f" [!] {feature}: p-value = {metrics['p_value']:.4e}")
else:
    print("\n[WARNING] Global drift test FAILED to detect the attack.")
    print("Because fraud is only 0.17% of the data, the macro-statistics barely changed.")
    print("This proves why statistical drift detection alone is NOT enough for highly imbalanced data.")

Running Statistical Drift Detection (KS-Test)...

SYSTEM ALERT: 29 features have significantly drifted from the baseline distribution.

Drifted Features Details:
 [!] V1: p-value = 4.1280e-06
 [!] V2: p-value = 4.3206e-52
 [!] V3: p-value = 1.0900e-09
 [!] V4: p-value = 4.6765e-12
 [!] V5: p-value = 6.5409e-04
 [!] V6: p-value = 2.6991e-18
 [!] V7: p-value = 7.8054e-09
 [!] V8: p-value = 9.9599e-18
 [!] V9: p-value = 3.8473e-11
 [!] V10: p-value = 7.0213e-24
 [!] V11: p-value = 9.4571e-07
 [!] V12: p-value = 2.3341e-30
 [!] V13: p-value = 1.8080e-24
 [!] V14: p-value = 2.4078e-09
 [!] V16: p-value = 2.4898e-16
 [!] V17: p-value = 2.6619e-03
 [!] V18: p-value = 1.1880e-08
 [!] V19: p-value = 1.7380e-05
 [!] V20: p-value = 6.5395e-09
 [!] V21: p-value = 6.1269e-49
 [!] V22: p-value = 1.0606e-38
 [!] V23: p-value = 9.3218e-12
 [!] V24: p-value = 4.8344e-04
 [!] V25: p-value = 2.8524e-27
 [!] V26: p-value = 4.6517e-18
 [!] V27: p-value = 2.6605e-18
 [!] V28: p-value = 1.6573e-08
 [!] Scale

**The Active Learning Safety Net**

In [ ]:
print("--- POST-DRIFT ACTIVE LEARNING ROUTER ---")
hitl_metrics_drifted = get_hitl_queue_metrics(y_test, y_pred_proba_drifted)

print(f"Transactions Flagged for Review (Uncertainty 0.4-0.6): {hitl_metrics_drifted['queue_size']}")
print(f"Actual Fraud Rescued by HITL: {hitl_metrics_drifted['actual_fraud_caught_by_hitl']}")

print("\nEngineering Conclusion:")
print("When the scammers shifted tactics, the model didn't just confidently fail.")
print("The attack pushed the model's probabilities into the uncertainty threshold, automatically routing the new attack vectors to human analysts before the money was lost.")

--- POST-DRIFT ACTIVE LEARNING ROUTER ---


NameError: name 'get_hitl_queue_metrics' is not defined